# 02 · Post-inclusion portfolio

This prototype builds a monthly portfolio for **1975–2025** from the provisional additions register in `01_inventory.ipynb`. Run all cells from the repository root or the `notebooks/` directory. The full series is available as `portfolio_monthly`.

At each month-end, form an equal-weighted basket of stocks that joined the S&P 500 during the preceding twelve months and are still eligible to hold. A March addition first contributes its April return. It can stay in the basket through the following March. A delisting ends the position after the delisting month. Additions documented in 1974 initialize the January 1975 basket.

This implements the calendar-time portfolio in Section 3.2 of the project plan. The long basket tests post-inclusion underperformance. The gross market-hedged short and multifactor regression extend that test. Net returns, the migration comparison, and the separate daily event study are further steps.

In [ ]:
from pathlib import Path

from IPython.utils.capture import capture_output
import numpy as np
import pandas as pd

inventory_path = Path("01_inventory.ipynb")
if not inventory_path.is_file():
    inventory_path = Path("notebooks") / "01_inventory.ipynb"

with capture_output(stdout=False, stderr=False):
    get_ipython().run_line_magic("run", f'"{inventory_path.resolve().as_posix()}"')

months = pd.PeriodIndex(calendar_dates["index_monthly"].dt.to_period("M"), name="month").sort_values()

## Build the monthly holdings

Each position is keyed by `PERMNO, month`. A second addition during the holding window extends the stock's eligibility at its usual weight. Recent additions can contribute in the observable months through December 2025.

In [ ]:
event_key = ["permno", "effective_date"]
warmup_start = sample_start - pd.DateOffset(years=1)
warmup = historical_spells.loc[
    historical_spells["start"].ge(warmup_start) & historical_spells["start"].lt(sample_start),
    ["permno", "start"],
].rename(columns={"start": "effective_date"})
cohorts = pd.concat([warmup, events[event_key]], ignore_index=True).drop_duplicates(event_key)
unmatched_events = int(events["permno"].isna().sum())
cohorts = cohorts.dropna(subset=["permno"]).astype({"permno": "int64"})

delistings = pd.read_csv(
    paths["crsp_delist"], usecols=["permno", "delistingdt"], parse_dates=["delistingdt"]
).drop_duplicates()
cohorts = pd.merge_asof(
    cohorts.sort_values("effective_date"),
    delistings.sort_values("delistingdt"),
    left_on="effective_date",
    right_on="delistingdt",
    by="permno",
    direction="forward",
)

holdings = cohorts.merge(pd.DataFrame({"holding_month": range(1, 13)}), how="cross")
holdings["month"] = holdings["effective_date"].dt.to_period("M") + holdings["holding_month"]
delist_month = holdings["delistingdt"].dt.to_period("M")
in_calendar = holdings["month"].isin(months)
still_held = delist_month.isna() | holdings["month"].le(delist_month)
holdings = (
    holdings.loc[in_calendar & still_held]
    .sort_values("effective_date")
    .drop_duplicates(["permno", "month"], keep="last")
)

## Monthly returns and coverage

The notebook uses CRSP monthly total returns (`MthRet`). Flags `G`, `M`, and `V` indicate that the delisting return is missing or partial. If any holding has a missing or incomplete return, the notebook leaves the portfolio return blank for that month. This avoids reallocating weights based on data availability after the fact. [CRSP flag definitions](https://www.crsp.org/wp-content/uploads/appendix/FlagType_DE.html).

In [ ]:
eligible_ids = holdings["permno"].unique()
monthly_returns = pd.concat(
    chunk.loc[chunk["PERMNO"].isin(eligible_ids)]
    for chunk in pd.read_csv(
        paths["crsp_monthly"],
        usecols=["PERMNO", "MthCalDt", "MthRet", "MthDelFlg"],
        parse_dates=["MthCalDt"],
        chunksize=500_000,
    )
).rename(columns={"PERMNO": "permno", "MthRet": "stock_return", "MthDelFlg": "delist_flag"})
monthly_returns["month"] = monthly_returns.pop("MthCalDt").dt.to_period("M")
monthly_returns = monthly_returns.drop_duplicates()

panel = holdings.merge(monthly_returns, on=["permno", "month"], how="left", validate="one_to_one")
panel["partial_delist"] = panel["delist_flag"].isin(["G", "M", "V"])
panel["stock_return"] = panel["stock_return"].mask(panel["partial_delist"])

portfolio_monthly = panel.groupby("month").agg(
    n_positions=("permno", "size"),
    n_returns=("stock_return", "count"),
    n_partial_delists=("partial_delist", "sum"),
    gross_return=("stock_return", "mean"),
).reindex(months)
counts = ["n_positions", "n_returns", "n_partial_delists"]
portfolio_monthly[counts] = portfolio_monthly[counts].fillna(0).astype(int)
portfolio_monthly["n_missing_returns"] = portfolio_monthly["n_positions"] - portfolio_monthly["n_returns"]
portfolio_monthly["gross_return"] = portfolio_monthly["gross_return"].where(
    portfolio_monthly["n_missing_returns"].eq(0)
)
portfolio_monthly["period"] = pd.cut(
    portfolio_monthly.index.year,
    bins=[1974, 1999, 2009, 2017, 2025],
    labels=["1975–1999", "2000–2009", "2010–2017", "2018–2025"],
)

print(f"{unmatched_events} additions omitted because they lack a PERMNO.")
display(portfolio_monthly.groupby("period", observed=True).agg(
    months=("gross_return", "size"),
    usable_months=("gross_return", "count"),
    missing_stock_returns=("n_missing_returns", "sum"),
))
display(portfolio_monthly.tail(12))

## Estimate a preliminary alpha

As in the *Factor models* lecture, the CAPM regresses the basket's return above the risk-free rate on the excess return of the U.S. market. The intercept is the monthly alpha. The notebook estimates it for the full sample and each of the four subperiods. The regression uses months with complete portfolio returns.

Place the original `F-F_Research_Data_Factors_CSV.zip` archive in `Data/` to populate `alpha_summary`. The portfolio series runs independently of the archive. [Download the Kenneth French monthly factors](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_CSV.zip).

In [ ]:
def estimate_capm(sample):
    sample = sample.dropna(subset=["gross_return", "risk_free", "market_excess"])
    y = sample["gross_return"] - sample["risk_free"]
    x = np.column_stack([np.ones(len(sample)), sample["market_excess"]])
    if len(sample) < 3 or np.linalg.matrix_rank(x) < 2:
        return {"months_used": len(sample), "alpha_monthly_pct": np.nan, "beta": np.nan}
    alpha, beta = np.linalg.lstsq(x, y, rcond=None)[0]
    return {"months_used": len(sample), "alpha_monthly_pct": alpha * 100, "beta": beta}


alpha_summary = None
factor_path = data_dir / "F-F_Research_Data_Factors_CSV.zip"
if factor_path.is_file():
    market_factors = read_monthly_factors(factor_path, ["Mkt-RF", "RF"]).rename(
        columns={"Mkt-RF": "market_excess", "RF": "risk_free"},
    )
    regression_data = portfolio_monthly.join(market_factors, validate="one_to_one")
    if regression_data[["market_excess", "risk_free"]].isna().any().any():
        raise ValueError("The factor archive must cover every month from 1975 through 2025.")
    alpha_summary = pd.DataFrame.from_dict(
        {"1975–2025": estimate_capm(regression_data)}
        | {
            period: estimate_capm(sample)
            for period, sample in regression_data.groupby("period", observed=True)
        },
        orient="index",
    )
    display(alpha_summary.round(4))
else:
    print("To estimate alpha, place F-F_Research_Data_Factors_CSV.zip in Data/ and rerun this cell.")

## Model a market-hedged short

The gross spread is long $1 of the CRSP value-weighted U.S. market and short $1 of the additions basket each month. Returns are measured per dollar of short exposure. The market leg uses `vwretd`, which includes distributions. The two legs have equal dollar notionals and the regression reports the remaining market beta. A month with an unavailable basket return stays blank.

This spread measures modeled profit and loss before trading, financing, and stock-borrow costs. [CRSP monthly variable definitions](https://wrds-www.wharton.upenn.edu/demo/crsp/form/).

In [ ]:
market_total = pd.read_csv(
    paths["index_monthly"], usecols=["mthcaldt", "vwretd"], parse_dates=["mthcaldt"],
)
market_total["month"] = market_total.pop("mthcaldt").dt.to_period("M")
market_total = market_total.set_index("month").rename(columns={"vwretd": "market_total_return"})

hedged_monthly = portfolio_monthly[["period", "n_positions", "gross_return"]].rename(
    columns={"gross_return": "long_basket_total_return"}
).join(market_total, validate="one_to_one")
if hedged_monthly["market_total_return"].isna().any():
    raise ValueError("The CRSP market series must cover the entire portfolio calendar.")
hedged_monthly["gross_hedged_short"] = (
    hedged_monthly["market_total_return"] - hedged_monthly["long_basket_total_return"]
)
display(hedged_monthly.tail(12))

## Estimate multifactor alpha

Run `03_strategy_inputs.ipynb` to prepare `monthly_ff5_momentum.csv`. The regression uses the five Fama–French factors and momentum. For the long basket, subtract the risk-free rate. For the self-financing long-short spread, regress the return difference directly.

Report monthly alpha, its t-statistic, and an approximate 95% confidence interval. Use Newey–West standard errors with Bartlett weights and 12 monthly lags, chosen for the twelve-month holding window. The regression uses observed returns and the standard errors preserve the calendar gap in January 1984. [Newey and West](https://www.nber.org/papers/t0055).

In [ ]:
factor_columns = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom"]


def estimate_multifactor_alpha(sample, return_column, lags=12):
    calendar = pd.period_range(sample.index.min(), sample.index.max(), freq="M")
    sample = sample.reindex(calendar)
    observed = sample[[return_column, *factor_columns]].notna().all(axis=1).to_numpy()
    outcome = sample.loc[observed, return_column].to_numpy()
    design = np.column_stack([np.ones(len(outcome)), sample.loc[observed, factor_columns]])
    coefficients, _, rank, _ = np.linalg.lstsq(design, outcome, rcond=None)
    n_obs, n_parameters = design.shape
    if n_obs <= n_parameters or rank < n_parameters:
        raise ValueError("The multifactor regression needs more observations and full-rank factors.")

    # Keep the residual score on the monthly calendar so missing returns do not compress lags.
    scores = np.zeros((len(calendar), n_parameters))
    scores[observed] = design * (outcome - design @ coefficients)[:, None]
    score_cov = scores.T @ scores
    lags = min(lags, len(calendar) - 1)
    for lag in range(1, lags + 1):
        cross_cov = scores[lag:].T @ scores[:-lag]
        score_cov += (1 - lag / (lags + 1)) * (cross_cov + cross_cov.T)
    bread = np.linalg.inv(design.T @ design)
    covariance = bread @ score_cov @ bread * n_obs / (n_obs - n_parameters)

    alpha = coefficients[0]
    alpha_se = np.sqrt(max(0.0, covariance[0, 0]))
    return {
        "months_used": n_obs,
        "alpha_monthly_pct": 100 * alpha,
        "nw_se_monthly_pct": 100 * alpha_se,
        "alpha_ci95_low_pct": 100 * (alpha - 1.96 * alpha_se),
        "alpha_ci95_high_pct": 100 * (alpha + 1.96 * alpha_se),
        "t_stat": alpha / alpha_se if alpha_se > 0 else np.nan,
        "market_beta": coefficients[1],
    }


multifactor_alpha_summary = None
prepared_factors = data_dir / "prepared" / "monthly_ff5_momentum.csv"
if prepared_factors.is_file():
    factors = pd.read_csv(prepared_factors, usecols=["month", *factor_columns, "RF"])
    factors.index = pd.PeriodIndex(factors.pop("month"), freq="M", name="month")
    multifactor_data = hedged_monthly.join(factors, validate="one_to_one")
    if multifactor_data[[*factor_columns, "RF"]].isna().any().any():
        raise ValueError("The prepared factors must cover the entire portfolio calendar.")
    multifactor_data["long_basket_excess_return"] = (
        multifactor_data["long_basket_total_return"] - multifactor_data["RF"]
    )

    samples = [("1975–2025", multifactor_data)]
    samples.extend(multifactor_data.groupby("period", observed=True))
    portfolios = {
        "Long additions": "long_basket_excess_return",
        "Gross hedged short": "gross_hedged_short",
    }
    multifactor_alpha_summary = pd.DataFrame([
        {"portfolio": portfolio, "period": period, **estimate_multifactor_alpha(sample, return_column)}
        for portfolio, return_column in portfolios.items()
        for period, sample in samples
    ]).set_index(["portfolio", "period"])
    display(multifactor_alpha_summary.round(4))
else:
    print("Run 03_strategy_inputs.ipynb with the five-factor and momentum archives to prepare the multifactor input.")